## XGBoost

### Import packages

In [99]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (roc_auc_score, f1_score,
                             classification_report, roc_curve)
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier
import shap
warnings.filterwarnings("ignore")

### Data preprocessing

In [6]:
os.getcwd()

'/Users/kirawei/Library/Mobile Documents/com~apple~CloudDocs/data-science/bnpl-prediction/src/model'

In [89]:
# LOAD
df = pd.read_csv("../../data/data_merged.csv", low_memory=False)

# exclude meta and target variables
EXCLUDE = ["BNPL1","BNPL3","Unnamed: 0","weight","year",
           "BNPL4_a","BNPL4_b","BNPL4_c","BNPL4_d","BNPL4_e","BNPL4_f",
           "BNPL1A","BNPL5"]

In [90]:

# ENCODING  (shared ordinal / binary / nominal maps)
ordinal_maps = {
    "ppfs1482": {"Very poor":1,"Poor":2,"Fair":3,"Good":4,"Excellent":5,
                 "Don\u2019t know":np.nan},
    "A6":       {"Not confident":1,"Somewhat confident":2,"Very confident":3,
                 "Don\u2019t know":np.nan},
    "ppinc7":   {"Less than $10,000":1,"$10,000 to $24,999":2,
                 "$25,000 to $49,999":3,"$50,000 to $74,999":4,
                 "$75,000 to $99,999":5,"$100,000 to $149,999":6,
                 "$150,000 or more":7},
    "ppeducat": {"No high school diploma or GED":1,
                 "High school graduate (high school diploma or the equivalent GED)":2,
                 "Some college or Associate\u2019s degree":3,
                 "Bachelor\u2019s degree or higher":4},
    "I20":      {"Less than your income":1,"The same as your income":2,
                 "More than your income":3},
    "C4A":      {"Never carried an unpaid balance (always pay in full)":1,
                 "Once":2,"Some of the time":3,"Most or all of the time":4},
    "ppfsasset":{"Under $50,000":1,"$50,000 - $99,999":2,
                 "$100,000 - $249,999":3,"$250,000 - $499,999":4,
                 "$500,000 - $999,999":5,"$1,000,000 or more":6,
                 "Not sure":np.nan},
    "INF4":     {"Much worse":1,"Somewhat worse":2,"Little or no effect":3,
                 "Somewhat better":4,"Much better":5},
    "C3P":      {"Did not pay or paid less than the minimum payment on at least one card":1,
                 "Paid at least the minimum payment on all credit cards":2,
                 "Did not use any of my credit cards so had no balances":3},
}

def encode(data):
    """Encode a dataframe slice. Returns encoded df, does not touch EXCLUDE cols."""
    d = data.copy()
    for col, m in ordinal_maps.items():
        if col in d.columns:
            d[col] = d[col].map(m)
    for c in [c for c in d.columns if c not in EXCLUDE
              and d[c].dropna().isin(["Yes","No","5 or more","'1'"]).all()]:
        d[c] = d[c].map({"Yes":1,"No":0, "5 or more":5, "'1'":1})
    nominal = [c for c in ["ppethm","ppgender","ppemploy","ppmarit5"]
               if c in d.columns]
    d = pd.get_dummies(d, columns=nominal, drop_first=True, dtype=float)
    return d

In [94]:
def prepare(data, label):
    print(f"\n── {label} ──")
    feat_cols = [c for c in data.columns if c not in EXCLUDE]

    # Encode
    working   = encode(data[feat_cols + ["BNPL1","BNPL3","weight"]].copy())
    feat_cols = [c for c in working.columns
                 if c not in ["BNPL1","BNPL3","weight"]]

    # Force all feature columns to numeric — catches any remaining strings
    # '1','2','5 or more' etc. → numeric; unparseable strings → NaN (XGBoost handles)
    for col in feat_cols:
        working[col] = pd.to_numeric(working[col], errors='coerce')
        if working[col].isin(['5 or more']).any():
            working[col] = working[col].replace({'5 or more': 5})

    # No scaling — not needed for XGBoost
    X = working[feat_cols].values

    # BNPL1 — adoption
    y1 = (working["BNPL1"] == "Yes").astype(int).values
    w1 = working["weight"].values

    # BNPL3 — delinquency (BNPL users only)
    mask = working["BNPL1"] == "Yes"
    y3   = (working.loc[mask, "BNPL3"] == "Yes").astype(int).values
    X3   = X[mask]
    w3   = w1[mask]

    print(f"  Adoption model  — N={len(y1):,}  positive={y1.mean():.1%}")
    print(f"  Delinquency model — N={len(y3):,}  positive={y3.mean():.1%}")
    return X, y1, w1, X3, y3, w3, feat_cols
#    return working

#working_A = prepare(df, "Model A — Full 2022-2025")
X_A, y1_A, w1_A, X3_A, y3_A, w3_A, feats_A = prepare(df, "Model A — Full 2022-2025")
X_B, y1_B, w1_B, X3_B, y3_B, w3_B, feats_B = prepare(df[df["year"]==2025].copy(), "Model B — 2025 only")


── Model A — Full 2022-2025 ──
  Adoption model  — N=48,294  positive=13.3%
  Delinquency model — N=6,407  positive=20.2%

── Model B — 2025 only ──
  Adoption model  — N=12,932  positive=15.5%
  Delinquency model — N=2,003  positive=24.3%


### XGBoost fit

In [96]:
def fit_xgboost(X, y, weights, label):
    cv    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scale_pos_weight = (y==0).sum()/(y==1).sum() # handle inblanced classes
    model = GridSearchCV(
        XGBClassifier(
            scale_pos_weight=scale_pos_weight,
            eval_metric='auc',
            max_iter=5000,
            random_state=42,
            n_jobs=-1),
        param_grid={
            'max_depth': [3, 5],
            'learning_rate': [0.05, 0.1],
            'n_estimators': [100, 200],
            },
        cv=cv, scoring='roc_auc', n_jobs=-1
    )

    model.fit(X, y, sample_weight=weights)

    best_model = model.best_estimator_
    y_prob     = best_model.predict_proba(X)[:, 1]
    y_pred     = best_model.predict(X)
    auc        = roc_auc_score(y, y_prob, sample_weight=weights)
    f1         = f1_score(y, y_pred, sample_weight=weights)

    print(f"\n  {label}")
    print(f"    Best params={model.best_params_} | AUC={auc:.4f} | F1={f1:.4f}")
    print(classification_report(y, y_pred, sample_weight=weights,
                                target_names=["No","Yes"], digits=3))
    return best_model, auc, f1

print("\n══ Fitting models ══")
m_adopt_A, auc_a1, f1_a1 = fit_xgboost(X_A,  y1_A, w1_A, "Adoption — Full")
m_adopt_B, auc_b1, f1_b1 = fit_xgboost(X_B,  y1_B, w1_B, "Adoption — 2025")
m_delin_A, auc_a3, f1_a3 = fit_xgboost(X3_A, y3_A, w3_A, "Delinquency — Full")
m_delin_B, auc_b3, f1_b3 = fit_xgboost(X3_B, y3_B, w3_B, "Delinquency — 2025")


══ Fitting models ══

  Adoption — Full
    Best params={'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200} | AUC=0.8122 | F1=0.4422
              precision    recall  f1-score   support

          No      0.948     0.718     0.817 41392.78809999911
         Yes      0.311     0.764     0.442 6902.1511999999975

    accuracy                          0.724 48294.93929999911
   macro avg      0.630     0.741     0.630 48294.93929999911
weighted avg      0.857     0.724     0.763 48294.93929999911


  Adoption — 2025
    Best params={'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200} | AUC=0.8306 | F1=0.4890
              precision    recall  f1-score   support

          No      0.944     0.718     0.816 10800.668399999904
         Yes      0.355     0.786     0.489 2132.2774000000027

    accuracy                          0.729 12932.945799999907
   macro avg      0.650     0.752     0.652 12932.945799999907
weighted avg      0.847     0.729     0.762 12932.94579999990

### AUC-ROC

In [97]:
BLUE = "#185FA5"; RED = "#A32D2D"; TEAL = "#0F6E56"; GRAY = "#4A4A4A"
# ROC curves — all 4 on one chart
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, pairs, title in [
    (axes[0], [
        (m_adopt_A, X_A,  y1_A, w1_A, f"Full 2022-2025 (AUC={auc_a1:.3f})", BLUE),
        (m_adopt_B, X_B,  y1_B, w1_B, f"2025 only (AUC={auc_b1:.3f})",       TEAL),
    ], "ROC — BNPL Adoption (BNPL1)"),
    (axes[1], [
        (m_delin_A, X3_A, y3_A, w3_A, f"Full 2022-2025 (AUC={auc_a3:.3f})", BLUE),
        (m_delin_B, X3_B, y3_B, w3_B, f"2025 only (AUC={auc_b3:.3f})",        TEAL),
    ], "ROC — BNPL Delinquency (BNPL3, users only)"),
]:
    for model, X, y, w, label, col in pairs:
        fpr, tpr, _ = roc_curve(y, model.predict_proba(X)[:,1], sample_weight=w)
        ax.plot(fpr, tpr, color=col, lw=2, label=label)
    ax.plot([0,1],[0,1], color=GRAY, ls="--", lw=1, label="Random")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(title)
    ax.legend(fontsize=9)

plt.tight_layout()
fig.savefig("../../img/models/xgboost_roc_curves.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved → xgboost_svc_roc_curves.png")


  Saved → xgboost_svc_roc_curves.png


### SHAP

In [101]:

def plot_shap(model, X, feat_cols, label, filename):
    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)

    # ── Bar plot (mean absolute SHAP — magnitude only, sorted) ───
    fig, ax = plt.subplots(figsize=(10, 8))
    shap.summary_plot(
        shap_values, X,
        feature_names=feat_cols,
        plot_type="bar",
        max_display=20,
        show=False,
        plot_size=None,
        color="#185FA5"
    )
    plt.title(f"SHAP Feature Importance — {label}\n"
              f"Mean |SHAP| = average impact on model output", fontsize=12, fontweight="bold")
    plt.xlabel("Mean |SHAP value|", fontsize=11)
    plt.tight_layout()
    plt.savefig(f"../../img/SHAP/{filename}_bar.png", dpi=150, bbox_inches="tight")
    plt.close()

    # ── Beeswarm plot (magnitude + direction) ────────────────────
    fig, ax = plt.subplots(figsize=(10, 8))
    shap.summary_plot(
        shap_values, X,
        feature_names=feat_cols,
        plot_type="dot",      # beeswarm — shows direction + magnitude
        max_display=20,
        show=False,
        plot_size=None
    )
    plt.title(f"SHAP Summary — {label}\n"
              f"Red = pushes toward Yes | Blue = pushes toward No",
              fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"../../img/SHAP/{filename}_beeswarm.png", dpi=150, bbox_inches="tight")
    plt.close()

    print(f"  Saved → shap_outputs/{filename}_bar.png")
    print(f"  Saved → shap_outputs/{filename}_beeswarm.png")

print("Generating SHAP plots...")
plot_shap(m_adopt_A, X_A,  feats_A, "Adoption — Full 2022-2025",  "XGBoost_SHAP_adoption_full")
plot_shap(m_adopt_B, X_B,  feats_B, "Adoption — 2025 only",       "XGBoost_SHAP_adoption_2025")
plot_shap(m_delin_A, X3_A, feats_A, "Delinquency — Full 2022-2025","XGBoost_SHAP_delinquency_full")
plot_shap(m_delin_B, X3_B, feats_B, "Delinquency — 2025 only",    "XGBoost_SHAP_delinquency_2025")
print("Done.")

Generating SHAP plots...
  Saved → shap_outputs/XGBoost_SHAP_adoption_full_bar.png
  Saved → shap_outputs/XGBoost_SHAP_adoption_full_beeswarm.png
  Saved → shap_outputs/XGBoost_SHAP_adoption_2025_bar.png
  Saved → shap_outputs/XGBoost_SHAP_adoption_2025_beeswarm.png
  Saved → shap_outputs/XGBoost_SHAP_delinquency_full_bar.png
  Saved → shap_outputs/XGBoost_SHAP_delinquency_full_beeswarm.png
  Saved → shap_outputs/XGBoost_SHAP_delinquency_2025_bar.png
  Saved → shap_outputs/XGBoost_SHAP_delinquency_2025_beeswarm.png
Done.
